In [1]:
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
df = pd.read_csv('house_price_prediction_dataset.csv')

In [3]:
df.shape

(5000, 14)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 14 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   property_type               5000 non-null   str    
 1   neighborhood                5000 non-null   str    
 2   square_feet                 4879 non-null   float64
 3   lot_square_feet             4588 non-null   float64
 4   bedrooms                    4822 non-null   float64
 5   bathrooms                   4741 non-null   float64
 6   year_built                  4727 non-null   float64
 7   condition                   4649 non-null   str    
 8   heating_type                4253 non-null   str    
 9   parking_spaces              4598 non-null   float64
 10  has_garden                  4767 non-null   str    
 11  renovated                   4699 non-null   str    
 12  distance_to_city_center_km  4789 non-null   float64
 13  sale_price_usd              5000 non-null   

In [5]:
df.describe()

,square_feet,lot_square_feet,bedrooms,bathrooms,year_built,parking_spaces,distance_to_city_center_km,sale_price_usd
count,4879.000000,4588.000000,4822.000000,4741.000000,4727.000000,4598.000000,4789.000000,5000.000000
mean,1876.759787,5608.369878,3.436748,2.376819,1981.957267,1.559809,6.971622,452508.586800
std,639.047207,3098.176587,1.401262,0.914840,24.848297,0.825293,4.643813,139966.649805
min,450.000000,800.000000,1.000000,1.000000,1940.000000,0.000000,0.200000,45000.000000
25%,1438.000000,3270.000000,2.000000,1.500000,1961.000000,1.000000,3.500000,355822.500000
50%,1867.000000,5195.000000,3.000000,2.500000,1981.000000,2.000000,6.000000,451563.000000
75%,2303.500000,7570.750000,4.000000,3.000000,2004.000000,2.000000,9.400000,547531.000000
max,4162.000000,18604.000000,7.000000,5.000000,2025.000000,3.000000,35.000000,977217.000000


In [6]:
df.isnull().sum()

property_type                   0
neighborhood                    0
square_feet                   121
lot_square_feet               412
bedrooms                      178
bathrooms                     259
year_built                    273
condition                     351
heating_type                  747
parking_spaces                402
has_garden                    233
renovated                     301
distance_to_city_center_km    211
sale_price_usd                  0
dtype: int64

In [7]:
# 2. How are prices distributed?
print(df["sale_price_usd"].describe())

count      5000.000000
mean     452508.586800
std      139966.649805
min       45000.000000
25%      355822.500000
50%      451563.000000
75%      547531.000000
max      977217.000000
Name: sale_price_usd, dtype: float64


In [8]:
# 3. Does size drive price? (the single strongest signal usually)
print(df[["square_feet", "sale_price_usd"]].corr())

                square_feet  sale_price_usd
square_feet         1.00000         0.89985
sale_price_usd      0.89985         1.00000


In [9]:
# Make sure 'None' in heating_type is treated as a real word, not a blank
df['heating_type'] = df['heating_type'].astype(str).replace('nan', np.nan)

In [10]:
# Create house age and remove the source year column
if "year_built" not in df:
    raise KeyError("The dataset must contain a year_built column")

df["house_age"] = 2026 - df["year_built"]
df = df.drop(columns="year_built")

In [11]:
print(df["condition"].dtype)
print(df["condition"].unique())

str
<StringArray>
['Good', 'Fair', 'Very Good', 'Excellent', nan, 'Poor']
Length: 6, dtype: str


In [12]:
# Encode condition as an ordered numeric scale while preserving missing values
condition_map = {"Poor": 1, "Fair": 2, "Good": 3, "Very Good": 4, "Excellent": 5}
unexpected_conditions = set(df["condition"].dropna().unique()) - set(condition_map)
if unexpected_conditions:
    raise ValueError(f"Unexpected condition labels: {sorted(unexpected_conditions)}")
df["condition"] = df["condition"].map(condition_map)

In [13]:
# Encode Yes/No columns
yes_no_map = {"Yes": 1, "No": 0}
for column in ("has_garden", "renovated"):
    unexpected_values = set(df[column].dropna().unique()) - set(yes_no_map)
    if unexpected_values:
        raise ValueError(f"Unexpected values in {column}: {sorted(unexpected_values)}")
    df[column] = df[column].map(yes_no_map)

In [14]:
x = df.drop("sale_price_usd", axis=1)
y = df["sale_price_usd"]

In [15]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [16]:
# Group columns by the preprocessing they need
numeric_features = [
    "square_feet", "lot_square_feet", "bedrooms", "bathrooms",
    "parking_spaces", "distance_to_city_center_km", "house_age", "condition",
]
categorical_features = [
    "property_type", "neighborhood", "heating_type", "has_garden", "renovated",
]

In [17]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

In [18]:
preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_transformer, numeric_features),
    ("categorical", categorical_transformer, categorical_features),
])

In [19]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
    )),
])

In [20]:
print("Training the model...")
model.fit(x_train, y_train)
print("Training complete")

Training the model...


Training complete


In [21]:
predictions = model.predict(x_test)
print(predictions)

[686017.76 272196.77 677772.59 571728.36 424964.69 402829.   413115.33
 307527.99 357412.95 438976.99 477852.53 472064.   250739.11 420563.97
 374666.19 329331.04 317301.35 462665.48 234461.54 327260.36 389309.24
 515000.07 406702.91 360392.62 401756.35 529757.59 391257.9  191642.11
 316381.07 696166.94 368712.03 283225.61 685676.34 518528.44 508709.1
 350035.24 664554.97 547837.67 440354.2  313996.12 294955.32 374050.06
 577326.54 533565.31 268501.94 591802.15 284209.48 464262.19 517731.82
 761943.47 506970.84 439067.21 355165.12 464222.58 521610.01 578482.98
 581692.72 722368.35 508754.74 469946.08 393617.81 687154.06 349776.77
 319172.9  570636.16 703951.83 446935.38 831486.86 454156.97 277644.74
 362104.81 406706.96 291359.54 540634.05 412520.37 425395.8  257578.49
 614036.92 431151.1  556967.52 517420.05 334205.11 486713.76 378538.3
 401780.25 349410.86 348895.17 293897.03 145335.8  455430.91 376199.64
 436742.39 438887.29 334333.94 456062.19 547635.18 217357.31 407591.27
 640548.

In [22]:
mae = mean_absolute_error(y_test, predictions)

In [23]:
print(mae)

36521.892719999996


In [24]:
r2 = r2_score(y_test, predictions)
print(r2)

0.8954539566249439


In [25]:
print("--- RESULTS ---")
print(f"On average, the model is off by: ${mae:,.2f}")
print(f"R2 score: {r2:.3f} (closer to 1.0 is better)")

--- RESULTS ---
On average, the model is off by: $36,521.89
R2 score: 0.895 (closer to 1.0 is better)


In [26]:
joblib.dump(model, "house_price_model.pkl")
print("Model saved as house_price_model.pkl")

Model saved as house_price_model.pkl
